# Manual de bases de datos vectoriales
## Fase 2 — Embeddings locales: de texto a vectores reales

**Objetivo de esta fase:** pasar de los vectores inventados de la fase 1 a **vectores reales calculados a partir de texto**, usando un modelo de embeddings que corre **en tu propia computadora** (sin API, sin cuenta, sin llave). Todo lo aprendido en la fase 1 (norma, producto punto, coseno, normalizar) se aplica aquí directamente.

**Qué vas a aprender**

| # | Tema |
|---|------|
| 0 | Precauciones de seguridad (**léelas antes de instalar o descargar nada**) |
| 1 | Preparar el entorno e instalar `sentence-transformers` |
| 2 | Qué es un modelo de embeddings y qué significa "local" |
| 3 | Descargar y cargar el modelo, y **verificar qué archivos bajaron** |
| 4 | El primer embedding real |
| 5 | ¿Vienen normalizados? Lo medimos |
| 6 | Varias frases a la vez: arrays de 2 dimensiones |
| 7 | Matriz de similitud coseno y mapa de calor (`imshow`) |
| 8 | Experimentos: paráfrasis, idiomas, negaciones, números, truncado |
| 9 | Resumen |

### Cómo usar este manual

Igual que la fase 1: todo el código viene completo, las celdas dependen de las anteriores, y cada concepto sigue el orden *idea → código → qué observar*. Todas las gráficas terminan con `plt.show()` para que el código se pueda copiar a un `.py`.

### Qué validé y qué NO validé (léelo, es importante)

- **Validé la lógica del código** (formas de los arrays, la verificación de la caché, las operaciones con matrices, las gráficas) ejecutándolo en mi entorno con un **modelo simulado**, porque desde mi entorno no puedo descargar el modelo real.
- **NO pude ver los resultados reales del modelo.** Cuando el notebook diga "qué observar", son **expectativas mías basadas en cómo se entrenan estos modelos**, no resultados que yo haya visto con este modelo. **Tú serás el primero en verlos.** Si algo contradice lo que escribo, anótalo y dímelo: es exactamente la información que necesito.
- Las celdas que comprueban **identidades matemáticas** (por ejemplo, que `distancia² = 2 − 2·coseno` con vectores normalizados) deben dar `True` sin importar el modelo. Si alguna da `False`, avísame, porque significa que hay un error mío.

---
# 0. Precauciones de seguridad

Vas a descargar un archivo de internet (el modelo) e instalar varias librerías. Esto es normal y de riesgo bajo si se hace con cuidado, pero **no es riesgo cero**. Estas son las precauciones, cada una con su motivo.

## 1. Dónde ejecutarlo

- En tu **PC con Windows 11**, dentro del entorno virtual `.venv`.
- **No** en el servidor de SQL Server de Extral ni en una sesión donde tengas credenciales de producción a la mano.
- En esta fase solo usamos **textos ficticios**. Ningún dato real de Extral hasta que decidamos cómo tratarlo.

*Por qué:* si algo saliera mal con un paquete o un archivo descargado, el daño queda limitado a un entorno desechable y sin datos sensibles.

## 2. Usa el nombre exacto del modelo oficial

El modelo oficial es **`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`**, publicado bajo la organización `sentence-transformers`. En mi búsqueda aparecieron también copias y versiones ajustadas de **terceros** con nombres casi idénticos (por ejemplo bajo `ciCic/...` o `Sprylab/...`). Esas **no** son el modelo oficial. Copia el nombre desde este notebook (variable `MODELO`) y no lo escribas de memoria.

## 3. Nunca uses `trust_remote_code=True`

Ese parámetro permite que un repositorio de Hugging Face **ejecute su propio código Python** en tu máquina. Este modelo no lo necesita. En el código fuente de `sentence-transformers` 6.1.0 verifiqué que el valor por defecto es `False`; **no lo cambies**.

## 4. Verifica el formato de los archivos de pesos

Existen dos formatos comunes para guardar un modelo:

- **`.safetensors`**: guarda solo números (los pesos). No ejecuta código al cargarse.
- **`.bin` / `.pt` (pickle)**: un formato que **puede ejecutar código arbitrario** al cargarse. (Esto es conocimiento general mío, no salió de mi búsqueda.)

Según lo que vi en la página oficial del modelo, el repositorio incluye `model.safetensors` (471 MB, marcado como "Safe" por Hugging Face), y en versiones anteriores del repositorio también aparece `pytorch_model.bin` (marcado como pickle). La marca "Safe" es un **escaneo automático**, no una garantía.

*Por eso la sección 3 incluye una celda que lista qué archivos quedaron realmente en tu caché.* No sé con certeza cuál de los dos usa tu versión de las librerías, así que lo comprobamos en lugar de suponerlo.

> **Regla:** si en tu caché ves **solo** un `.bin` y **ningún** `.safetensors`, detente y avísame antes de seguir.

## 5. Qué sale de tu equipo y qué no

- **Tus textos no salen de tu máquina** al generar embeddings: el modelo corre local. (Es la naturaleza de la inferencia local; yo no audité el código completo de las librerías.)
- **La descarga inicial sí contacta a Hugging Face.**
- **Telemetría:** la propia librería `huggingface_hub` documenta que envía datos de uso para rastrear el uso de las librerías de Hugging Face, y que se desactiva con la variable de entorno `HF_HUB_DISABLE_TELEMETRY=1` (también queda desactivada en modo offline). La celda de abajo la desactiva.
- **Modo offline:** una vez descargado el modelo, la variable `HF_HUB_OFFLINE=1` hace que las librerías **no intenten conectarse a internet**. Es opcional y se explica en la sección 3.

## 6. Cadena de suministro de `pip`

- Instala **solo** los nombres exactos: `numpy`, `matplotlib`, `ipykernel`, `sentence-transformers`. Revisa la ortografía: publicar paquetes con nombres casi iguales a los populares es una práctica conocida de ataque.
- Instálalos **dentro del `.venv`** (con el entorno activado), nunca en el Python global.
- `sentence-transformers` trae muchas dependencias (torch, transformers, scikit-learn, scipy...). Después de instalar, `pip list` te muestra todo lo que entró.
- Opcional: guardar tus versiones exactas con `pip freeze` te permite reconstruir el mismo entorno después. Es un archivo que tú decides crear.

## 7. Límites de lo que verifiqué

No pude auditar los archivos del modelo. Lo que sé viene de la página oficial, de su licencia (Apache 2.0) y de que es un modelo público y muy usado. El riesgo residual es **bajo, pero no cero**.

---

Ejecuta la siguiente celda **antes de importar cualquier librería de Hugging Face**: desactiva la telemetría y revisa el entorno. Si reinicias el kernel del notebook, vuelve a ejecutarla primero.

In [ ]:
import os
import sys

# Debe ejecutarse ANTES de importar sentence_transformers / huggingface_hub
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# --- 1. ¿Estoy dentro de un entorno virtual? ---
# Dentro de un venv, sys.prefix apunta al venv y sys.base_prefix al Python "base".
estoy_en_venv = sys.prefix != sys.base_prefix
print("¿Dentro de un entorno virtual? ->", estoy_en_venv)
print("Intérprete:", sys.executable)
if not estoy_en_venv:
    print("AVISO: no estás en un entorno virtual. Selecciona el kernel del .venv antes de instalar nada.")

# --- 2. El modelo, con su nombre exacto y oficial ---
MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
ORGANIZACION_OFICIAL = "sentence-transformers"

organizacion = MODELO.split("/")[0]
print()
print("Modelo      :", MODELO)
print("Organización:", organizacion)
print("¿Organización oficial? ->", organizacion == ORGANIZACION_OFICIAL)

# --- 3. Telemetría desactivada para esta sesión ---
print()
print("HF_HUB_DISABLE_TELEMETRY =", os.environ.get("HF_HUB_DISABLE_TELEMETRY"))

Deberías ver `True` en la organización oficial y `HF_HUB_DISABLE_TELEMETRY = 1`. Si el primer dato dice `False` (no estás en un `.venv`), corrígelo antes de continuar.

---
# 1. Preparar el entorno

Usamos **el mismo `.venv` de la fase 1**: ya tiene numpy y matplotlib. Solo falta instalar `sentence-transformers`.

## Instalación

Con el entorno activado (debe verse `(.venv)` en la terminal; si no, repite la activación de la fase 1):

```powershell
pip install sentence-transformers
```

**Qué esperar** (esto es aproximado, verifícalo):

- Es una instalación **grande**: `sentence-transformers` arrastra `torch` (la librería de redes neuronales), `transformers` y otras. Puede tardar varios minutos y ocupar de cientos de MB a un par de GB.
- **No necesitas tarjeta gráfica.** En Windows, PyPI hospeda versiones de `torch` solo para CPU, así que el `pip install` normal debería instalar la versión de CPU. Mis fuentes lo indican así, pero no lo probé en tu equipo.

**Compatibilidad con tu Python 3.12:** al escribir este manual, la versión más reciente de `sentence-transformers` que encontré en PyPI era la **6.1.0**, y su metadata declara `Python >= 3.10`, así que acepta tu 3.12. También declara que necesita `torch >= 2.2`, `transformers >= 5` y `huggingface-hub` 1.x. **Si `pip` falla**, copia el mensaje de error completo y lo resolvemos; no lo adivines.

> **Nota sobre versiones:** entre versiones mayores de una librería, algunas funciones cambian de nombre. Por eso, en el código de este manual verás un pequeño `try/except` que prueba el nombre nuevo y, si no existe, el viejo. Así el notebook funciona aunque tengas una versión distinta a la mía.

Ejecuta esta celda para confirmar que todo se instaló.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sentence_transformers
import torch
import huggingface_hub

print("numpy                :", np.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("torch                :", torch.__version__)
print("huggingface_hub      :", huggingface_hub.__version__)
print("¿GPU disponible?     :", torch.cuda.is_available())

Lo normal en una PC sin tarjeta gráfica dedicada es `¿GPU disponible? False`: el modelo correrá en el procesador, y para un modelo de este tamaño es suficiente.

---
# 2. Qué es un modelo de embeddings

## Idea intuitiva

Un **modelo de embeddings** es una **red neuronal ya entrenada** que recibe texto y devuelve un vector. Se entrenó para que **textos con significado parecido produzcan vectores cercanos**.

## El recorrido de una frase (simplificado)

1. **Tokenización:** la frase se parte en *tokens*, que son pedazos de palabras. "extrusora" podría convertirse en varios tokens.
2. **Red neuronal (transformer):** procesa los tokens y produce un vector por cada uno, teniendo en cuenta el contexto de los demás.
3. **Pooling:** esos vectores por token se combinan en **un solo vector** para toda la frase. Según las descripciones públicas que encontré, este modelo usa *mean pooling*, es decir, un promedio.
4. **Resultado:** un vector de 384 números que representa la frase completa.

## Datos del modelo que usaremos

`paraphrase-multilingual-MiniLM-L12-v2`, según las páginas públicas que consulté (algunas son de terceros, como la cifra de parámetros; verifícalas si te importan):

| Propiedad | Valor |
|-----------|-------|
| Dimensiones del vector | 384 |
| Idiomas | más de 50 (incluye español) |
| Parámetros | ~118 millones |
| Longitud máxima de texto | **128 tokens** (lo que pase de ahí se **trunca**) |
| Licencia | Apache 2.0 |

El nombre indica su especialidad: **"paraphrase"** = está entrenado para reconocer que dos frases dicen lo mismo con palabras distintas; **"multilingual"** = funciona en muchos idiomas a la vez.

## Qué significa "local"

- **Local:** el modelo es un archivo que descargas **una vez** y queda en tu disco. Generar embeddings se ejecuta en tu procesador. Gratis, sin límites, sin cuenta, y tus textos no viajan.
- **Por API** (fase 6): envías el texto a un servicio externo, que te devuelve el vector. Necesita cuenta, llave, tiene límites y tus textos salen de tu equipo.

## Qué NO hace (importante)

Un modelo de embeddings **no "entiende" ni verifica verdades**. Mide qué tan **parecidos** son dos textos según lo que aprendió durante el entrenamiento. Eso es útil, pero tiene límites que veremos con experimentos en la sección 8 (negaciones, números, orden de palabras).

---
# 3. Descargar y cargar el modelo

## Cargar el modelo

La primera vez que ejecutes la siguiente celda, la librería **descargará el modelo** desde Hugging Face (alrededor de 0.5 GB según lo que vi en la página oficial) y lo guardará en una carpeta de caché. Las siguientes veces lo carga desde el disco y es mucho más rápido.

Observa que **no** pasamos `trust_remote_code`: usamos el valor por defecto (`False`).

> **Si te aparece un aviso sobre "symlinks" o enlaces simbólicos** (es algo que puede pasar en Windows): es un aviso sobre cómo se guarda la caché en disco, no sobre seguridad. La librería incluso tiene una variable (`HF_HUB_DISABLE_SYMLINKS_WARNING`) para silenciarlo. Si el aviso es otro, o hay un error de conexión, copia el mensaje completo y lo vemos.
>
> **Si la descarga falla por la red**, es posible que la red de la empresa bloquee Hugging Face. No lo puedo saber desde aquí.

In [ ]:
import time
from sentence_transformers import SentenceTransformer

inicio = time.perf_counter()
modelo = SentenceTransformer(MODELO)     # sin trust_remote_code: se usa el valor por defecto (False)
segundos = time.perf_counter() - inicio

print(f"Modelo cargado en {segundos:.1f} s")

## Datos del modelo cargado

Comprobamos que lo que cargaste coincide con la ficha pública. Usamos `try/except` por el cambio de nombre entre versiones que mencioné en la sección 1.

In [ ]:
try:
    dimensiones = modelo.get_embedding_dimension()            # nombre en versiones recientes
except AttributeError:
    dimensiones = modelo.get_sentence_embedding_dimension()   # nombre en versiones anteriores

print("Dimensiones del vector :", dimensiones)
print("Longitud máx. (tokens) :", modelo.max_seq_length)
print("Dispositivo            :", modelo.device)
print("Clase                  :", type(modelo).__name__)

**Qué observar:** según la ficha del modelo deberías ver **384** dimensiones y **128** tokens. El dispositivo normalmente será `cpu`. Si alguno de los dos números es distinto, anótalo: afecta a los experimentos de las secciones siguientes.

## Verificar qué archivos quedaron en tu caché (precaución #4)

`huggingface_hub` incluye una función, `scan_cache_dir()`, que revisa la carpeta de caché y te dice qué descargó y cuánto pesa. La usamos para comprobar **qué formato de pesos** bajó tu equipo.

Primero mostramos dónde está la caché, y luego revisamos el modelo.

In [ ]:
from huggingface_hub import constants, scan_cache_dir

print("Carpeta de caché de Hugging Face:", constants.HF_HUB_CACHE)

info_cache = scan_cache_dir()

repo_modelo = None
for repo in info_cache.repos:
    if repo.repo_id == MODELO:
        repo_modelo = repo

if repo_modelo is None:
    print()
    print("No encontré el modelo en la caché estándar.")
    print("Puede estar en otra carpeta (si definiste HF_HOME o HF_HUB_CACHE). Dímelo y lo revisamos.")
else:
    print()
    print(f"Modelo: {repo_modelo.repo_id}")
    print(f"Tamaño total en disco: {repo_modelo.size_on_disk / 1e6:.0f} MB")

    for revision in repo_modelo.revisions:
        archivos = sorted(f.file_name for f in revision.files)
        print()
        print(f"Revisión {revision.commit_hash[:8]} - {len(archivos)} archivos:")
        for f in sorted(revision.files, key=lambda x: x.file_name):
            print(f"   {f.file_name:<28} {f.size_on_disk / 1e6:>9.2f} MB")

        tiene_safetensors = any(nombre.endswith(".safetensors") for nombre in archivos)
        extensiones_pickle = (".bin", ".pt", ".pth", ".pkl", ".ckpt")
        tiene_pickle = any(nombre.endswith(extensiones_pickle) for nombre in archivos)

        print()
        print("¿Hay archivo .safetensors? ->", tiene_safetensors)
        print("¿Hay archivo tipo pickle?  ->", tiene_pickle)

        if tiene_pickle and not tiene_safetensors:
            print("ALTO: solo hay formato pickle. Detente y avísame antes de seguir.")

**Cómo leer el resultado:**

| Lo que ves | Qué significa |
|------------|---------------|
| `.safetensors` sí, pickle no | Lo ideal. |
| `.safetensors` sí, pickle sí | Aceptable para este repositorio oficial. No sé con certeza cuál de los dos cargó tu versión de la librería, pero el formato seguro está disponible. Anótalo. |
| Pickle sí, `.safetensors` no | **Detente y avísame.** |

Además de los pesos, es normal ver archivos pequeños de configuración (`config.json`, `modules.json`, etc.) y el archivo del tokenizador. Si ves algo que no reconoces, dímelo.

## Modo offline (opcional)

Una vez que el modelo ya está en tu caché, puedes impedir que las librerías intenten conectarse a internet. Define la variable de entorno **antes de arrancar el notebook** (o al inicio de la primera celda, antes de cualquier import de Hugging Face):

```python
import os
os.environ["HF_HUB_OFFLINE"] = "1"
```

Confirmé que esa variable existe en el código de `huggingface_hub`. Con ella activa, si el modelo **no** está en caché, la carga fallará en lugar de descargarlo, lo cual es justo lo que queremos si buscamos que nada salga a la red.

---
# 4. El primer embedding real

Con `modelo.encode(texto)` convertimos una frase en un vector. Usaremos frases **ficticias**.

In [ ]:
frase = "La máquina de extrusión se detuvo por una falla en el motor."

vector = modelo.encode(frase)

print("Tipo  :", type(vector))
print("shape :", vector.shape)    # (384,)  -> un vector de 384 números
print("dtype :", vector.dtype)    # tipo de los números
print()
print("Primeros 8 números:")
print(vector[:8])

**Qué observar:**

- `shape` debe ser `(384,)`: un vector de una sola dimensión con 384 números (lo que aprendimos en la fase 1: `shape (n,)`).
- `dtype` probablemente sea `float32` (decimales de 32 bits, más ligeros que los `float64` de la fase 1). Lo que veas es el dato.
- **Los números sueltos no significan nada para un humano.** No existe "la posición 57 significa maquinaria". Lo útil es comparar vectores completos.

## ¿El mismo texto da siempre el mismo vector?

Comprobémoslo, usando `np.isclose` / `np.allclose` (compara con tolerancia, como vimos en la fase 1 por los decimales).

In [ ]:
vector_otra_vez = modelo.encode(frase)
print("¿Mismo vector al repetir? ->", np.allclose(vector, vector_otra_vez))

otra_frase = "Hoy preparé una ensalada de pollo con aguacate."
vector_otra_frase = modelo.encode(otra_frase)
print("¿Misma frase vs. otra frase distinta? ->", np.allclose(vector, vector_otra_frase), "(se espera False)")

Se espera `True` en la primera comprobación (mismo texto, mismo modelo, mismo vector) y `False` en la segunda. Si la primera diera `False`, avísame.

---
# 5. ¿Vienen normalizados? Lo medimos

En la fase 1 prometí que **mediríamos y no supondríamos** si los vectores de un modelo vienen normalizados (norma 1). Lo hacemos ahora.

Primero recreamos las funciones de la fase 1 (copiadas tal cual), porque este notebook es independiente:

In [ ]:
def distancia_euclidiana(a, b):
    '''Distancia en línea recta entre dos vectores. Menor = más cercanos.'''
    return np.linalg.norm(a - b)


def similitud_coseno(a, b):
    '''Coseno del ángulo entre dos vectores, entre -1 y 1. Mayor = más parecidos.'''
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))


def normalizar(v):
    '''Devuelve el vector con la misma dirección pero longitud 1.'''
    return v / np.linalg.norm(v)


norma = np.linalg.norm(vector)
print("Norma del vector que devuelve encode() por defecto:", norma)
print("¿Norma ≈ 1? ->", np.isclose(norma, 1.0, atol=1e-3))

**Qué observar:** el resultado depende del modelo, y **yo no lo sé de antemano**:

- Si `¿Norma ≈ 1?` es `True`, este modelo ya entrega vectores normalizados por defecto.
- Si es `False`, la norma será otro número, y conviene normalizar antes de comparar.

(La tolerancia `atol=1e-3` es porque con `float32` la norma de un vector "de longitud 1" puede salir como 0.9999999 o 1.0000001.)

## Normalizar al generar el embedding

`encode` tiene un parámetro `normalize_embeddings=True` que entrega los vectores ya con norma 1. Comprobamos que hace **lo mismo que nuestra función `normalizar`** de la fase 1:

In [ ]:
vector_n = modelo.encode(frase, normalize_embeddings=True)

print("Norma con normalize_embeddings=True:", np.linalg.norm(vector_n))
print("¿Coincide con normalizar() de la fase 1? ->", np.allclose(vector_n, normalizar(vector), atol=1e-5))

La norma debe ser ≈ 1 y la comparación debe dar `True`: es la misma operación (dividir entre la norma).

## Comprobar con vectores reales lo que demostramos en la fase 1

En la fase 1 demostramos que, con vectores de norma 1:

$$\|a-b\|^2 = 2 - 2\cos(\theta) \qquad\text{y}\qquad \cos(\theta) = a\cdot b$$

Ahora lo comprobamos con vectores **reales** de dos frases parecidas. Estas comprobaciones deben dar `True` **con cualquier modelo**, porque son identidades matemáticas.

In [ ]:
frase_a = "La máquina de extrusión se detuvo por una falla en el motor."
frase_b = "Se paró la extrusora porque el motor dejó de funcionar."

va = modelo.encode(frase_a, normalize_embeddings=True)
vb = modelo.encode(frase_b, normalize_embeddings=True)

coseno = similitud_coseno(va, vb)
producto_punto = va @ vb
distancia_al_cuadrado = distancia_euclidiana(va, vb) ** 2

print("similitud coseno      :", coseno)
print("producto punto        :", producto_punto)
print("distancia²            :", distancia_al_cuadrado)
print("2 - 2 * coseno        :", 2 - 2 * coseno)
print()
print("¿coseno = producto punto?      ->", np.isclose(coseno, producto_punto, atol=1e-5))
print("¿distancia² = 2 - 2*coseno?    ->", np.isclose(distancia_al_cuadrado, 2 - 2 * coseno, atol=1e-4))

**Por qué importa:** con vectores normalizados, el coseno es **solo un producto punto**, que es una operación muy rápida. Y ordenar por menor distancia da el mismo orden que ordenar por mayor coseno. Las bases de datos vectoriales aprovechan esto. **A partir de aquí normalizaremos siempre** (`normalize_embeddings=True`).

---
# 6. Varias frases a la vez: arrays de 2 dimensiones

## El conjunto de frases ficticias

Armamos una lista de 9 frases con **relaciones intencionales** entre ellas. Los comentarios indican qué pretende cada una (son mi diseño, no un resultado del modelo):

In [ ]:
frases = [
    "La máquina de extrusión se detuvo por una falla en el motor.",        # 0  frase base
    "Se paró la extrusora porque el motor dejó de funcionar.",             # 1  paráfrasis de la 0
    "The extrusion machine stopped because of a motor failure.",           # 2  la 0 en inglés
    "La máquina de extrusión NO se detuvo; el motor funciona bien.",       # 3  la 0 con negación
    "El operador reportó una avería eléctrica en la línea de llenado.",    # 4  otra falla, tema cercano
    "Se necesita aumentar la velocidad de la línea de producción.",        # 5  producción, otro asunto
    "Hoy preparé una ensalada de pollo con aguacate para la comida.",      # 6  comida
    "La receta lleva pollo, aguacate y un poco de limón.",                 # 7  comida (parecida a la 6)
    "El equipo de fútbol ganó el partido con un gol al final.",            # 8  deportes
]

for i, f in enumerate(frases):     # enumerate da el índice y el elemento a la vez
    print(f"[{i}] {f}")

## Generar todos los embeddings de una vez

`encode` acepta una **lista** de frases y devuelve un solo array de **2 dimensiones**.

### Arrays de 2 dimensiones (matrices)

Hasta ahora un vector era un array de 1 dimensión, con `shape (384,)`. Una **matriz** es un array de 2 dimensiones: una tabla con **filas** y **columnas**. Aquí:

- **Cada fila** es el embedding de una frase.
- **Cada columna** es una de las 384 dimensiones.
- `shape` será `(9, 384)`: 9 filas (frases) × 384 columnas (dimensiones).

In [ ]:
embeddings = modelo.encode(frases, normalize_embeddings=True, show_progress_bar=False)

print("shape:", embeddings.shape)   # (9, 384)
print("ndim :", embeddings.ndim)    # 2 ejes: filas y columnas
print()

# Cómo "rebanar" una matriz: matriz[filas, columnas]
print("Fila 0 completa tiene shape:", embeddings[0].shape)           # una frase
print("Fila 0, primeras 5 columnas:", embeddings[0, :5])             # [fila, columnas 0 a 4]
print("Columna 0, todas las filas :", embeddings[:, 0].shape)        # un valor por frase
print("Filas 0 a 2, columnas 0 a 3:")
print(embeddings[:3, :4])

**La regla de las rebanadas:** `matriz[fila, columna]`, y `:` significa "todos". `embeddings[0]` es la fila 0; `embeddings[:, 0]` es la columna 0; `embeddings[:3, :4]` son las primeras 3 filas y 4 columnas.

Comprobación: la fila 0 de la matriz debe ser el mismo vector que generamos antes de forma individual (con una diferencia numérica mínima posible por el procesamiento en lote):

In [ ]:
print("¿Fila 0 ≈ vector individual normalizado? ->", np.allclose(embeddings[0], vector_n, atol=1e-5))
print("¿Todas las filas tienen norma ≈ 1?       ->", np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-4))

`np.linalg.norm(embeddings, axis=1)` calcula la norma **de cada fila** (`axis=1` significa "a lo largo de las columnas, fila por fila") y devuelve 9 números. Ambas comprobaciones deben dar `True`.

---
# 7. Matriz de similitud coseno

## Comparar todas contra todas con una sola operación

Queremos el coseno entre **cada par** de las 9 frases: 9 × 9 = 81 comparaciones. En vez de un ciclo, usamos **multiplicación de matrices**:

```python
S = embeddings @ embeddings.T
```

- `embeddings.T` es la **transpuesta**: voltea filas por columnas, así que `(9, 384)` pasa a `(384, 9)`.
- `@` entre `(9, 384)` y `(384, 9)` da una matriz `(9, 9)`.
- La entrada `S[i, j]` es el **producto punto de la fila `i` con la fila `j`**, es decir, la similitud coseno entre la frase `i` y la frase `j`, **porque los vectores tienen norma 1** (sección 5).

In [ ]:
S = embeddings @ embeddings.T

print("shape de S:", S.shape)   # (9, 9)

# Mostrar con 2 decimales para poder leerla
np.set_printoptions(precision=2, suppress=True, linewidth=160)
print(S)

Tres comprobaciones (deben dar `True` con cualquier modelo):

1. La diagonal (cada frase contra sí misma) debe valer ≈ 1.
2. La matriz es **simétrica**: `S[i, j] = S[j, i]` (el coseno no depende del orden).
3. Un par cualquiera calculado con nuestra función de la fase 1 coincide con la matriz.

In [ ]:
print("¿Diagonal ≈ 1?          ->", np.allclose(np.diag(S), 1.0, atol=1e-4))
print("¿Matriz simétrica?      ->", np.allclose(S, S.T, atol=1e-5))

i, j = 0, 6
print(f"¿S[{i},{j}] = similitud_coseno(fila {i}, fila {j})? ->",
      np.isclose(S[i, j], similitud_coseno(embeddings[i], embeddings[j]), atol=1e-5))

## Mapa de calor con `imshow`

Una matriz de 81 números es difícil de leer. Un **mapa de calor** pinta cada celda con un color según su valor. En matplotlib se hace con `ax.imshow(matriz)`:

- `imshow` dibuja la matriz como una imagen: cada celda `(fila, columna)` es un cuadro de color.
- `fig.colorbar(...)` agrega la barra que traduce colores a valores.
- Con `ax.text(...)` escribimos el número dentro de cada celda. Elegimos el color del texto (blanco o negro) según el fondo, para que se lea bien.

In [ ]:
n = len(frases)

fig, ax = plt.subplots(figsize=(8, 7))
imagen = ax.imshow(S, cmap="viridis", vmax=1.0)

# Etiquetas de ejes: el índice de cada frase
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels([str(i) for i in range(n)])
ax.set_yticklabels([str(i) for i in range(n)])
ax.set_xlabel("frase")
ax.set_ylabel("frase")
ax.set_title("Similitud coseno entre las 9 frases ficticias")

# Número dentro de cada celda; color del texto según qué tan claro/oscuro sea el fondo
umbral = (S.min() + 1.0) / 2
for fila in range(n):
    for col in range(n):
        color_texto = "black" if S[fila, col] > umbral else "white"
        ax.text(col, fila, f"{S[fila, col]:.2f}", ha="center", va="center",
                color=color_texto, fontsize=8)

fig.colorbar(imagen, ax=ax, label="similitud coseno")
plt.show()

# Leyenda con el texto de cada índice
for i, f in enumerate(frases):
    print(f"[{i}] {f}")

## Ordenar por parecido: una mini búsqueda semántica

Tomamos la frase 0 como "consulta" y ordenamos las demás de más a menos parecida. `np.argsort(x)` devuelve los **índices** que ordenarían `x` de menor a mayor; con un signo `-` delante, se invierte el orden (de mayor a menor).

Esto ya es, en pequeño, lo que hace una base de datos vectorial: **dada una consulta, devolver los textos más cercanos**.

In [ ]:
consulta = 0
orden = np.argsort(-S[consulta])   # índices de mayor a menor similitud con la consulta

print(f"Consulta: [{consulta}] {frases[consulta]}")
print()
for posicion in orden:
    print(f"{S[consulta, posicion]:6.3f}  [{posicion}] {frases[posicion]}")

## Qué observar (mis expectativas, NO verificadas con este modelo)

Por cómo se entrenan estos modelos, **esperaría** lo siguiente. Compáralo con lo que viste y anota las diferencias:

- La frase **1** (paráfrasis) entre las más cercanas a la **0**.
- La frase **2** (la misma idea en inglés) también cercana, si el modelo multilingüe funciona bien entre idiomas.
- Las frases **6 y 7** (comida) cercanas **entre sí** y **lejanas** de la 0.
- La frase **8** (fútbol) con valores bajos contra casi todas.
- La frase **3** (la negación de la 0): **sospecho que sale sorprendentemente cercana a la 0**, a pesar de decir lo contrario. Es una limitación conocida de este tipo de modelos, pero **no la verifiqué con este modelo**. Es justo lo que probamos en la sección 8.

Si tus resultados difieren, no es un error de tu código: es información sobre el modelo, y me interesa.

---
# 8. Experimentos

Aquí probamos los límites del modelo. En cada uno anoto qué espero, pero **el dato real es el que tú veas**.

## Una función auxiliar para comparar dos textos

In [ ]:
def comparar(texto_1, texto_2):
    '''Devuelve la similitud coseno entre dos textos (vectores normalizados).'''
    v1, v2 = modelo.encode([texto_1, texto_2], normalize_embeddings=True, show_progress_bar=False)
    return float(v1 @ v2)     # float(...) convierte el resultado a un número normal de Python

(`v1, v2 = ...` "desempaqueta" las dos filas de la matriz de 2×384 en dos vectores.)

## 8.1 Significado frente a palabras: sinónimos y polisemia

- **Sinónimos sin palabras en común:** "El coche no arranca." y "El automóvil no enciende." dicen lo mismo con palabras distintas. Como control, "El coche es rojo." comparte palabras pero habla de otra cosa.
- **Polisemia:** "banco" como asiento y "banco" como institución financiera.

In [ ]:
pares_significado = [
    ("El coche no arranca.", "El automóvil no enciende."),     # mismo sentido, palabras distintas
    ("El coche no arranca.", "El coche es rojo."),              # palabras en común, otro sentido
    ("Me senté en el banco del parque a descansar.", "El banco cerró temprano y no pude depositar."),  # polisemia
]

for t1, t2 in pares_significado:
    print(f"{comparar(t1, t2):6.3f}   {t1!r}  vs  {t2!r}")

**Expectativa (no verificada):** el primer par debería tener mayor similitud que el segundo, porque el modelo está hecho para captar significado y no solo palabras. El tercero es un buen test de si distingue los dos sentidos de "banco".

## 8.2 Idiomas

El modelo es multilingüe, así que la misma idea en español e inglés debería quedar cerca. Ya vimos el par 0-2 en la matriz; aquí otro caso:

In [ ]:
pares_idiomas = [
    ("Necesito revisar el reporte de producción.", "I need to review the production report."),
    ("Necesito revisar el reporte de producción.", "I love eating tacos on Sundays."),   # control: otro tema
]

for t1, t2 in pares_idiomas:
    print(f"{comparar(t1, t2):6.3f}   {t1!r}  vs  {t2!r}")

**Expectativa (no verificada):** el primer par claramente más alto que el segundo, aunque sean idiomas distintos.

## 8.3 Orden de palabras y negaciones

Aquí aparecen las limitaciones conocidas. Comparamos frases con **las mismas palabras pero significado opuesto o distinto**:

In [ ]:
pares_limites = [
    ("El perro mordió al hombre.", "El hombre mordió al perro."),                    # mismas palabras, distinto sentido
    ("El motor funciona bien.", "El motor no funciona bien."),                       # negación
    ("La máquina de extrusión se detuvo por una falla en el motor.",
     "La máquina de extrusión NO se detuvo; el motor funciona bien."),               # la negación de la sección 7
]

for t1, t2 in pares_limites:
    print(f"{comparar(t1, t2):6.3f}   {t1!r}  vs  {t2!r}")

**Expectativa (no verificada):** valores **altos** en los tres pares, aunque el significado cambie. Si es así, la lección es importante: **similitud alta no significa "dicen lo mismo" ni "es verdad"**. Mide cercanía de tema y de forma de decir las cosas, no lógica.

**Por qué importa en un contexto industrial:** "el motor funciona" y "el motor no funciona" pueden quedar casi iguales para el modelo. Un sistema de búsqueda semántica sobre reportes de paro debe tener esto en cuenta.

## 8.4 Números

En datos industriales los números importan. ¿Los distingue el modelo?

In [ ]:
pares_numeros = [
    ("La temperatura del horno es de 50 grados.", "La temperatura del horno es de 500 grados."),
    ("La temperatura del horno es de 50 grados.", "La temperatura del horno es de 51 grados."),
    ("La temperatura del horno es de 50 grados.", "La presión de la línea de aire es de 6 bar."),   # otro tema
]

for t1, t2 in pares_numeros:
    print(f"{comparar(t1, t2):6.3f}   {t1!r}  vs  {t2!r}")

**Expectativa (no verificada):** los dos primeros pares con valores muy altos, casi iguales entre sí, aunque 50 y 500 sean 10 veces distintos. Si es así, **los embeddings no son una buena herramienta para comparar cantidades**: para eso se usan las columnas numéricas de la base de datos y no el vector.

## 8.5 El límite de 128 tokens: truncado

El modelo solo lee los primeros **128 tokens** (según su ficha; lo verificamos con `modelo.max_seq_length`). Lo que sigue se **ignora en silencio**, sin error ni aviso.

El experimento: construimos un texto largo de relleno que supera el límite, y le agregamos una frase importante **al final**. Si el truncado funciona como describo, el vector del texto con la frase final debe ser **idéntico** al del texto sin ella, porque esa parte nunca se leyó.

Para contar tokens usamos el tokenizador del propio modelo. (Es posible que veas un aviso diciendo que la secuencia es más larga que el máximo del modelo: es esperado, justamente estamos probando eso. Si en cambio esta celda da un **error** al usar `modelo.tokenizer`, copia el mensaje: puede ser un cambio de la versión de la librería que no pude comprobar sin el modelo real.)

In [ ]:
import math

unidad = "La línea de producción opera con normalidad. "
cola = "Además, hay una fuga de aire comprimido en la prensa."

tokens_unidad = len(modelo.tokenizer(unidad)["input_ids"])
limite = modelo.max_seq_length

# Repetimos la unidad el DOBLE de veces necesarias para pasarnos del límite con holgura
repeticiones = 2 * math.ceil(limite / tokens_unidad)
relleno = unidad * repeticiones

texto_a = relleno             # solo relleno
texto_b = relleno + cola      # relleno + frase importante AL FINAL
texto_c = cola + relleno      # frase importante AL INICIO + relleno

for nombre, texto in [("A (solo relleno)", texto_a), ("B (relleno + cola)", texto_b), ("C (cola + relleno)", texto_c)]:
    n_tokens = len(modelo.tokenizer(texto)["input_ids"])
    print(f"{nombre:<22} -> {n_tokens} tokens (límite del modelo: {limite})")

In [ ]:
va = modelo.encode(texto_a, normalize_embeddings=True)
vb = modelo.encode(texto_b, normalize_embeddings=True)
vc = modelo.encode(texto_c, normalize_embeddings=True)

print("A vs B (frase importante al FINAL)  -> coseno:", float(va @ vb))
print("A vs C (frase importante al INICIO) -> coseno:", float(va @ vc))
print()
print("¿A y B tienen el mismo vector? ->", np.allclose(va, vb, atol=1e-6))

**Qué observar:**

- Si `A vs B` da coseno ≈ 1 y `¿A y B tienen el mismo vector?` da `True`, el truncado funciona como describí: la frase del final **no se leyó**. Si da `False`, mi explicación del truncado es incorrecta o incompleta: avísame.
- `A vs C` debería dar un coseno **claramente menor** que `A vs B`, porque la frase del inicio **sí** entra en los primeros 128 tokens y cambia el vector.

**Consecuencia práctica (la usaremos en las fases 3 y 7):** los documentos largos **no se pueden vectorizar enteros**. Hay que partirlos en fragmentos (en inglés, *chunks*) que quepan en el límite, y generar un vector por fragmento.

## 8.6 Velocidad: en lote o de uno en uno

`encode` con una lista procesa las frases en **lotes** (`batch_size`, por defecto 32), lo que normalmente es más eficiente que llamarlo frase por frase. Lo medimos con 45 frases. **Los tiempos dependen de tu equipo.**

In [ ]:
muchas_frases = frases * 5     # 9 frases x 5 = 45 frases

inicio = time.perf_counter()
_ = modelo.encode(muchas_frases, batch_size=32, show_progress_bar=False)
t_lote = time.perf_counter() - inicio

inicio = time.perf_counter()
for f in muchas_frases:
    _ = modelo.encode(f, show_progress_bar=False)
t_una_por_una = time.perf_counter() - inicio

print(f"En lote         : {t_lote:.2f} s para {len(muchas_frases)} frases")
print(f"Una por una     : {t_una_por_una:.2f} s para {len(muchas_frases)} frases")
print(f"Diferencia      : el lote fue {t_una_por_una / t_lote:.1f} veces más rápido en esta ejecución")

La conclusión práctica es **procesar siempre en lote** cuando haya muchos textos.

---
# 9. Resumen

## Tabla de referencia rápida

| Qué quieres hacer | Código |
|-------------------|--------|
| Cargar el modelo (oficial, sin `trust_remote_code`) | `SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")` |
| Embedding de una frase | `modelo.encode(frase)` → array `(384,)` |
| Embeddings de varias frases | `modelo.encode(lista)` → array `(n, 384)` |
| Vectores con norma 1 | `modelo.encode(..., normalize_embeddings=True)` |
| Similitud entre todas las frases | `S = E @ E.T` (con `E` normalizado) |
| Ordenar de mayor a menor similitud | `np.argsort(-S[i])` |
| Longitud máxima del modelo | `modelo.max_seq_length` |
| Revisar qué descargó tu equipo | `scan_cache_dir()` de `huggingface_hub` |

## Ideas clave

1. Un **embedding** es un vector que un modelo calcula a partir de texto; textos de significado parecido quedan cerca.
2. **"Local"** significa que el modelo corre en tu PC: sin API, sin cuenta, y tus textos no salen.
3. Con vectores **normalizados**, el coseno es un simple producto punto, y `S = E @ E.T` da todas las similitudes de golpe.
4. El modelo mide **parecido**, no verdad: negaciones, orden de palabras y números pueden engañarlo (según mi expectativa; **lo viste tú con tus datos**).
5. El límite de **128 tokens** obliga a partir los textos largos en fragmentos.
6. Procesar en **lote** es mucho más eficiente que frase por frase.

## Lista de seguridad (para recordar)

- Ejecutar en tu PC, dentro del `.venv`, con datos ficticios.
- Usar siempre el nombre oficial del modelo; nunca `trust_remote_code=True`.
- Verificar en la caché que existe `.safetensors`; si solo hay pickle, parar.
- Telemetría desactivada (`HF_HUB_DISABLE_TELEMETRY=1`); modo offline opcional (`HF_HUB_OFFLINE=1`).
- Instalar solo paquetes con nombre exacto y dentro del `.venv`.

## Lo que viene: Fase 3 — Búsqueda semántica en memoria

Con la matriz de similitud ya hicimos una mini búsqueda. En la fase 3 la hacemos "de verdad":

- Un conjunto más grande de documentos ficticios, partido en fragmentos.
- Una función `buscar(consulta, k)` que devuelve los `k` fragmentos más cercanos.
- Un **mapa 2D** de los textos, para ver cómo se agrupan por tema (aprenderemos a reducir 384 dimensiones a 2, con numpy).
- Todo todavía **sin base de datos**, para entender bien el mecanismo antes de pasarlo a PostgreSQL en la fase 4.

Cuando termines esta fase, cuéntame **qué resultados te salieron** en las secciones 7 y 8 (sobre todo negaciones, números y truncado), porque son los que decidirán cómo diseñamos las siguientes fases.